# 26 — Production I/O: Semi-Structured JSON, Excel Workbooks, & Modern Parquet
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to high-throughput data ingestion, flattening nested JSON hierarchies (`pd.json_normalize`), managing multi-sheet Excel workbooks, and benchmarking Parquet columnar storage.*

---

## 📌 Executive Summary & Interview Expectations
In real-world data platforms, data rarely arrives as a pristine CSV. Interviewers test your ability to ingest and export across heterogeneous serialization formats:
1. **Semi-Structured JSON Normalization**: Unpacking deeply nested dictionaries and arrays with `pd.json_normalize()`, handling missing keys (`laureates`), and metadata attribution.
2. **JSON Serialization Orientations**: Grasping the architectural differences between `records`, `split`, `index`, and `columns` orientations.
3. **Multi-Sheet Excel Management**: Loading multiple tabs simultaneously (`sheet_name=None`) into dictionary structures and exporting formatted workbooks via `pd.ExcelWriter`.
4. **Columnar Parquet vs Row-Based Formats**: Why modern analytical warehouses (Snowflake, BigQuery, Spark) standardize on Parquet for 10x-50x speedups, column pruning, and schema enforcement.

## 1. Semi-Structured JSON & `pd.json_normalize()`

### 💡 The Missing Key Trap in Real APIs
- In the Nobel Prize dataset, prizes in certain years (e.g. 1940-1942 during WWII) were not awarded, so the `"laureates"` key is **missing**.
- Blindly calling `pd.json_normalize(nobel['prizes'], record_path='laureates')` crashes with `KeyError: "Key 'laureates' not found"`.
- **Production Solution**: Use `errors='ignore'` or pre-populate missing keys with `p.setdefault('laureates', [])`!

In [1]:
import os
import json
import numpy as np
import pandas as pd

# Load Nobel Prize JSON
with open("nobel.json") as f:
    raw_nobel = json.load(f)

print(f"Loaded Nobel JSON with {len(raw_nobel['prizes'])} prize records.")

# Inspect a single prize entry
sample_prize = raw_nobel["prizes"][0]
print("Sample Prize Keys:", sample_prize.keys())
print("Sample Prize Category & Year:", sample_prize["category"], sample_prize["year"])

Loaded Nobel JSON with 646 prize records.
Sample Prize Keys: dict_keys(['year', 'category', 'laureates'])
Sample Prize Category & Year: chemistry 2019


In [2]:
# Flatten a single record
flat_single = pd.json_normalize(
    data=sample_prize,
    record_path="laureates",
    meta=["year", "category"]
)
print("Flattened 2019 Chemistry Laureates:")
display(flat_single)

Flattened 2019 Chemistry Laureates:


,id,firstname,surname,motivation,share,year,category
0,976,John,Goodenough,"""for the development of lithium-ion batteries""",3,2019,chemistry
1,977,M. Stanley,Whittingham,"""for the development of lithium-ion batteries""",3,2019,chemistry
2,978,Akira,Yoshino,"""for the development of lithium-ion batteries""",3,2019,chemistry


In [3]:
# Robust Pipeline: Flattening entire collection with missing-key safeguarding
# Ensure every prize entry contains 'laureates' key (defaulting to empty list if missing)
prizes_cleaned = [
    {**p, "laureates": p.get("laureates", [])}
    for p in raw_nobel["prizes"]
]

all_laureates = pd.json_normalize(
    data=prizes_cleaned,
    record_path="laureates",
    meta=["year", "category"],
    errors="ignore"
)

print(f"Successfully Flattened Universe: {len(all_laureates)} total Nobel Laureate records!")
display(all_laureates[["year", "category", "firstname", "surname", "share"]].head(5))

Successfully Flattened Universe: 950 total Nobel Laureate records!


,year,category,firstname,surname,share
0,2019,chemistry,John,Goodenough,3
1,2019,chemistry,M. Stanley,Whittingham,3
2,2019,chemistry,Akira,Yoshino,3
3,2019,economics,Abhijit,Banerjee,3
4,2019,economics,Esther,Duflo,3


## 2. JSON Serialization Orientations

### 💡 JSON Orient Cheat Sheet
| Orient | Structure | Best Use Case |
| :--- | :--- | :--- |
| `'records'` | `[{col: val}, {col: val}]` | REST APIs, document databases (MongoDB) |
| `'split'` | `{'columns': [...], 'index': [...], 'data': [...]}` | Network transmission with minimal payload overhead |
| `'index'` | `{index_label: {col: val}}` | Fast key-based lookup dictionaries |
| `'columns'` | `{col: {index_label: val}}` | Columnar data transmission |

In [4]:
sample_mini = all_laureates.head(2)[["year", "category", "surname"]]

print("--- Orient = 'records' (Standard API Payload) ---")
print(sample_mini.to_json(orient="records", indent=2))

print("\n--- Orient = 'split' (Payload Compression - Columns separate from Data) ---")
print(sample_mini.to_json(orient="split", indent=2))

--- Orient = 'records' (Standard API Payload) ---
[
  {
    "year":"2019",
    "category":"chemistry",
    "surname":"Goodenough"
  },
  {
    "year":"2019",
    "category":"chemistry",
    "surname":"Whittingham"
  }
]

--- Orient = 'split' (Payload Compression - Columns separate from Data) ---
{
  "columns":[
    "year",
    "category",
    "surname"
  ],
  "index":[
    0,
    1
  ],
  "data":[
    [
      "2019",
      "chemistry",
      "Goodenough"
    ],
    [
      "2019",
      "chemistry",
      "Whittingham"
    ]
  ]
}


## 3. CSV I/O & Large-File Streaming via `chunksize`

### 💡 Interview Tip: Never Load Multi-Gigabyte CSVs with Blind `pd.read_csv`
When files exceed available RAM:
```python
chunks = pd.read_csv('massive_file.csv', chunksize=100_000)
for chunk in chunks:
    # Process chunk in memory
```

In [5]:
# Stream NYC Baby Names in chunks of 5,000 rows
chunk_sizes = []
for chunk in pd.read_csv("baby_names.csv", chunksize=5000):
    chunk_sizes.append(len(chunk))

print(f"Processed Baby Names across {len(chunk_sizes)} chunks (Total rows: {sum(chunk_sizes)})")

Processed Baby Names across 5 chunks (Total rows: 24292)


## 4. Multi-Sheet Excel Workbooks: `sheet_name=None` & `ExcelWriter`

### 💡 Reading All Sheets at Once:
Passing `sheet_name=None` returns an **`OrderedDict` of DataFrames**, where keys are sheet names and values are the corresponding DataFrames!

In [6]:
# 1. Read single sheet
single_sheet = pd.read_excel("Single Worksheet.xlsx")
print(f"Single Sheet Loaded: {single_sheet.shape}")
display(single_sheet.head(2))

# 2. Read all sheets dynamically
all_sheets_dict = pd.read_excel("Multiple Worksheets.xlsx", sheet_name=None)
print("\nDiscovered Sheets in Workbook:", list(all_sheets_dict.keys()))

# 3. Export multiple DataFrames into a new multi-tab workbook
with pd.ExcelWriter("exported_summary.xlsx", engine="openpyxl") as writer:
    all_laureates.head(10).to_excel(writer, sheet_name="Top_Laureates", index=False)
    single_sheet.head(10).to_excel(writer, sheet_name="Single_Sample", index=False)

print("Exported multi-tab workbook 'exported_summary.xlsx' successfully!")

Single Sheet Loaded: (5, 4)


,First Name,Last Name,City,Gender
0,Brandon,James,Miami,M
1,Sean,Hawkins,Denver,M



Discovered Sheets in Workbook: ['Data 1', 'Data 2', 'Data 3']
Exported multi-tab workbook 'exported_summary.xlsx' successfully!


## 5. Modern Columnar Storage: Parquet vs CSV

### 🚨 Top Interview Topic: Why Parquet Outperforms CSV
1. **Columnar Layout**: Reading 2 columns from a 100-column Parquet file only reads 2% of the bytes from disk!
2. **Built-in Snappy Compression**: Files are typically 70-90% smaller on disk.
3. **Type Preservation**: Dtypes (`datetime64`, `int32`, `category`) are stored in file metadata; no string parsing needed on reload!

In [7]:
# Ingest and save as Parquet
baby_df = pd.read_csv("baby_names.csv")
baby_df.to_parquet("baby_names.parquet", engine="pyarrow", compression="snappy")

# Read back from Parquet
parquet_loaded = pd.read_parquet("baby_names.parquet", columns=["Year of Birth", "Child's First Name", "Count"])

print("Loaded from Parquet with column projection:")
print(f"Original columns: {baby_df.shape[1]} | Loaded columns: {parquet_loaded.shape[1]}")
display(parquet_loaded.head(3))

Loaded from Parquet with column projection:
Original columns: 6 | Loaded columns: 3


,Year of Birth,Child's First Name,Count
0,2011,GERALDINE,13
1,2011,GIA,21
2,2011,GIANNA,49


## 6. Real-World Challenge: Normalizing `tv_shows.json`

In [8]:
with open("tv_shows.json") as f:
    tv_data = json.load(f)

# Unpack episodes array per show, keeping show, runtime, and network as metadata
tv_episodes = pd.json_normalize(
    data=tv_data["shows"],
    record_path="episodes",
    meta=["show", "runtime", "network"]
)

print(f"Flattened TV Shows Matrix: {tv_episodes.shape}")
display(tv_episodes.head(5))

Flattened TV Shows Matrix: (482, 7)


,season,episode,name,air_date,show,runtime,network
0,1,1,Pilot,1993-09-11 01:00:00,The X-Files,60,FOX
1,1,2,Deep Throat,1993-09-18 01:00:00,The X-Files,60,FOX
2,1,3,Squeeze,1993-09-25 01:00:00,The X-Files,60,FOX
3,1,4,Conduit,1993-10-02 01:00:00,The X-Files,60,FOX
4,1,5,The Jersey Devil,1993-10-09 01:00:00,The X-Files,60,FOX


---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: Why does `pd.read_csv()` often produce different dtypes than when you saved the DataFrame?
**Answer**:
CSV is an un-typed text format. When exporting a DataFrame with `int32`, `category`, and `datetime64[ns]` columns to CSV, all values are serialized into plain ASCII strings.
When reading back via `pd.read_csv()`, Pandas uses type heuristics:
- `category` reverts back to `object` (string).
- Datetimes revert back to `object` unless `parse_dates` is explicitly configured.
- `int32` often gets upcast to `int64`.
*Parquet solves this completely by embedding the Apache Arrow schema directly in the file footer.*

---

### Q2: Advanced Interview Coding Challenge: Memory Footprint & Storage Benchmarking
**Challenge**:
Compare file size on disk between CSV and Parquet for the `baby_names` dataset, and calculate the compression ratio!

In [9]:
# Interview Solution: Storage Audit Script
csv_bytes = os.path.getsize("baby_names.csv")
parquet_bytes = os.path.getsize("baby_names.parquet")

compression_ratio = (1 - (parquet_bytes / csv_bytes)) * 100

print(f"CSV File Size:     {csv_bytes / 1024:,.1f} KB")
print(f"Parquet File Size: {parquet_bytes / 1024:,.1f} KB")
print(f"Disk Savings:      {compression_ratio:.1f}% space saved with Parquet!")

CSV File Size:     982.5 KB
Parquet File Size: 95.8 KB
Disk Savings:      90.3% space saved with Parquet!
